# Verify masked (de-identified) calls before vendor export

Samples N redacted calls from a run, plays the **redacted** audio next to the PII that
was supposed to be removed, records a pass/fail per call, and stages only the approved
ones into the `{call_id}/audio.opus` layout the S3 sync expects.

Run with the main venv kernel:

```bash
/mnt/amc-data/venvs/main/bin/python -m ipykernel install --user --name amc-main
```

**Order matters.** Cell 3 prints the real values of `redacted_status` and the
`pii_count` distribution before anything is sampled, so you set the filter in cell 4
from what is actually there. Nothing is uploaded from this notebook -- the last cell
prints the commands for you to run in a shell.

In [ ]:
# ---- 1. Config -------------------------------------------------------------
from pathlib import Path

RUN = "2022-full"          # or "2026-full"
RUN_ROOT = Path(f"/mnt/amc-data/amc-runs/{RUN}")

N_CALLS = 30               # calls to LISTEN TO
EXPORT_N = 20              # of those, the most that get staged for the vendor
SEED = 20260727            # change to draw a different sample; keep to reproduce one

# Skip long calls: you have to listen to a call end to end to claim you verified
# it, so a 40-minute one is not 8x the work of a 5-minute one, it is the reason
# the review never gets finished.
MAX_CALL_SEC = 360         # 6 minutes
MIN_CALL_SEC = 10          # a 3-second call proves nothing about masking

# Reading every shard to find 20 calls is wasteful. A call never splits across
# shards (its segments and its redacted audio live under the same shard), so a
# couple of shards is a valid draw -- raise for more spread across the corpus.
N_SHARDS = 3

# Where approved calls get staged. This directory is what you point `aws s3 sync`
# at, so its immediate children must be call_id dirs and nothing else.
STAGE_DIR = RUN_ROOT / "export" / "deid_results"

# Play the ORIGINAL un-redacted audio side by side. Off by default: that audio
# still contains the PII, which is the thing you are trying not to hand around.
# Turn on only when a call fails and you need to hear what should have been cut.
PLAY_ORIGINAL = False

print(f"run       : {RUN_ROOT}")
print(f"stage dir : {STAGE_DIR}")
print(f"sampling  : {N_CALLS} calls from {N_SHARDS} shard(s), seed {SEED}")
print(f"export    : up to {EXPORT_N} of the {N_CALLS} reviewed")

In [ ]:
# ---- 2. Imports and helpers ------------------------------------------------
import json
import random
import subprocess
from collections import Counter

import numpy as np
import pyarrow.compute as pc
import pyarrow.dataset as ds
from IPython.display import Audio, HTML, clear_output, display

WANT_COLUMNS = [
    "segment_id", "call_id", "channel", "year", "language",
    # source_duration_sec is the length of the CALL. duration_sec is one VAD
    # segment, and their sum is speech time only -- a 20-minute call with two
    # minutes of talking would pass a filter built on the latter.
    "start_sec", "end_sec", "duration_sec", "source_duration_sec",
    "final_transcript", "normalized_final_transcript",
    "pii_count", "pii_spans_json", "mask_intervals_json",
    "redacted_audio_path_abs", "redacted_status", "redacted_fallback_error",
    "segment_audio_path_abs", "source_path_abs",
    "model_agreement", "selected_model",
]

ALL_MANIFESTS = sorted(
    RUN_ROOT.glob("outputs/shard-*/manifests/all_segments.parquet"),
    key=lambda p: int(p.parts[-3].removeprefix("shard-")),
)
assert ALL_MANIFESTS, f"no manifests under {RUN_ROOT}"

_rng = random.Random(SEED)
MANIFESTS = _rng.sample(ALL_MANIFESTS, min(N_SHARDS, len(ALL_MANIFESTS)))

DATASET = ds.dataset([str(p) for p in MANIFESTS], format="parquet")
COLUMNS = [c for c in WANT_COLUMNS if c in DATASET.schema.names]
MISSING = [c for c in WANT_COLUMNS if c not in DATASET.schema.names]


def scan(filter_expr=None, columns=None):
    """Filtered read with predicate pushdown, so we never materialise 21M rows."""
    return DATASET.to_table(columns=columns or COLUMNS, filter=filter_expr).to_pylist()


def load_audio(path):
    """Decode to mono float32. soundfile handles ogg/opus; ffmpeg is the fallback."""
    path = str(path)
    try:
        import soundfile as sf
        data, sr = sf.read(path, dtype="float32", always_2d=False)
        if getattr(data, "ndim", 1) > 1:
            data = data.mean(axis=1)
        return data, sr
    except Exception:
        raw = subprocess.run(
            ["ffmpeg", "-v", "error", "-i", path, "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
            capture_output=True, check=True,
        ).stdout
        return np.frombuffer(raw, dtype=np.float32).copy(), 16000


def parse_json(value, default):
    if not value:
        return default
    try:
        return json.loads(value)
    except (TypeError, ValueError):
        return default


def hms(seconds):
    seconds = float(seconds or 0.0)
    return f"{int(seconds // 60):02d}:{seconds % 60:05.2f}"


print(f"manifests in run : {len(ALL_MANIFESTS)}")
print(f"sampling shards  : {[p.parts[-3] for p in MANIFESTS]}")
if MISSING:
    print(f"columns absent from this run's schema (skipped): {MISSING}")

In [ ]:
# ---- 3. What is actually in these shards -----------------------------------
# Three narrow columns only, so this stays cheap. Read the output before you set
# the filter in the next cell -- do not assume what "redacted" is called here.
probe_cols = [c for c in ("redacted_status", "pii_count", "call_id") if c in DATASET.schema.names]
probe = DATASET.to_table(columns=probe_cols).to_pydict()

n_rows = len(probe[probe_cols[0]])
statuses = Counter(str(s) for s in probe.get("redacted_status", []))
pii = [int(v or 0) for v in probe.get("pii_count", [])]

print(f"segments in sampled shards : {n_rows:,}")
print(f"distinct calls             : {len(set(probe['call_id'])):,}")
print()
print("redacted_status:")
for value, count in statuses.most_common():
    print(f"  {value!r:28s} {count:>10,}  ({count / n_rows * 100:5.2f}%)")
print()
if pii:
    with_pii = sum(1 for v in pii if v > 0)
    print(f"segments with pii_count > 0 : {with_pii:,}  ({with_pii / len(pii) * 100:.2f}%)")
    print(f"total PII spans             : {sum(pii):,}")
    print("pii_count histogram         :", dict(sorted(Counter(pii).items())[:8]), "...")

# Calls that actually had something redacted are the ones worth listening to;
# a call with zero PII proves nothing about the masking.
calls_with_pii = {
    c for c, p in zip(probe["call_id"], pii) if p > 0
} if pii else set()
print(f"\ncalls containing >=1 PII span : {len(calls_with_pii):,}")

In [ ]:
# ---- 4. Draw the sample ----------------------------------------------------
# Set this from the status values printed above. None = accept any status.
REQUIRE_STATUS = None      # e.g. "ok" / "redacted" / "masked"

selected_calls = sorted(calls_with_pii)
if not selected_calls:
    raise SystemExit("no calls with PII in the sampled shards -- raise N_SHARDS")
_rng.shuffle(selected_calls)
# Draw a surplus. REQUIRE_STATUS culls calls further down, and trimming to
# N_CALLS before that filter runs is what leaves you reviewing 13 calls when you
# asked for 20.
selected_calls = selected_calls[: N_CALLS * 4]

flt = pc.field("call_id").isin(selected_calls)
rows = scan(filter_expr=flt)

by_call = {}
for r in rows:
    by_call.setdefault(r["call_id"], []).append(r)
for segs in by_call.values():
    segs.sort(key=lambda r: (int(r.get("channel") or 0), float(r.get("start_sec") or 0.0)))

# Every segment of a call points at the same per-call redacted file.
def call_audio(segs):
    for r in segs:
        p = r.get("redacted_audio_path_abs")
        if p:
            return Path(p)
    return None

def call_length(segs):
    """Wall length of the call -- what you actually sit through.

    source_duration_sec describes the whole recording; falling back to the last
    segment end only under-reports by the trailing silence.
    """
    for r in segs:
        v = r.get("source_duration_sec")
        if v:
            return float(v)
    return max((float(r.get("end_sec") or 0.0) for r in segs), default=0.0)


summary = []
for call_id, segs in by_call.items():
    audio = call_audio(segs)
    statuses_here = {str(r.get("redacted_status")) for r in segs}
    summary.append({
        "call_id": call_id,
        "segments": len(segs),
        "pii_spans": sum(int(r.get("pii_count") or 0) for r in segs),
        "speech_sec": sum(float(r.get("duration_sec") or 0.0) for r in segs),
        "call_sec": call_length(segs),
        "status": ",".join(sorted(statuses_here)),
        "exists": bool(audio and audio.exists()),
        "size_kb": round(audio.stat().st_size / 1024, 1) if audio and audio.exists() else 0.0,
        "path": str(audio) if audio else "",
    })
if REQUIRE_STATUS is not None:
    summary = [s for s in summary if REQUIRE_STATUS in s["status"]]

# Length filter, on the call not the speech time. Verifying a call means hearing
# all of it, so cap the length rather than the count.
_before = len(summary)
summary = [s for s in summary if MIN_CALL_SEC <= s["call_sec"] <= MAX_CALL_SEC]
print(f"length filter {MIN_CALL_SEC}-{MAX_CALL_SEC}s ({MAX_CALL_SEC/60:.0f} min): "
      f"kept {len(summary)} of {_before} candidate calls")
# Only files that exist can be listened to or shipped.
summary = [s for s in summary if s["exists"]]

# Trim in the shuffled draw order, and only then sort for display. Taking the
# top N of a pii-sorted list would quietly turn the random sample into "the 20
# calls with the most PII" -- which reviews the easiest cases to spot and skips
# the sparse ones where a single missed span is the whole failure.
_draw_order = {call_id: i for i, call_id in enumerate(selected_calls)}
summary.sort(key=lambda d: _draw_order.get(d["call_id"], 1 << 30))
summary = summary[:N_CALLS]
summary.sort(key=lambda d: -d["pii_spans"])

if not summary:
    raise SystemExit(
        f"no calls between {MIN_CALL_SEC}s and {MAX_CALL_SEC}s with a redacted file. "
        "Raise MAX_CALL_SEC or N_SHARDS."
    )

BY_ID = {s["call_id"]: s for s in summary}

print(f"calls to review : {len(summary)}")
print(f"PII spans       : {sum(s['pii_spans'] for s in summary):,}")
print(f"total listening : {sum(s['call_sec'] for s in summary) / 60:.0f} min "
      f"(longest {max(s['call_sec'] for s in summary) / 60:.1f} min)")

# Plain text, no HTML table and no audio widgets. Copy an id into cell 6.
print("\n" + "-" * 64)
print(f"{'#':>3}  {'call_id':38}  {'min':>5} {'PII':>4} {'segs':>5}")
print("-" * 64)
for i, s in enumerate(summary):
    print(f"{i:>3}  {s['call_id']:38}  {s['call_sec']/60:>5.1f} {s['pii_spans']:>4} {s['segments']:>5}")
print("-" * 64)
print("\nCALL_IDS = [")
for s in summary:
    print(f'    "{s["call_id"]}",')
print("]")

In [ ]:
# ---- 5. Learn the PII / mask JSON shape ------------------------------------
# These two columns drive the whole review, and their schema differs between
# pipeline versions. Print one real example of each instead of guessing.
example = next(
    (r for segs in by_call.values() for r in segs if int(r.get("pii_count") or 0) > 0),
    None,
)
if example is None:
    print("no PII segment in the sample")
else:
    print("segment_id :", example["segment_id"])
    print("transcript :", repr(example.get("final_transcript"))[:300])
    print("\npii_spans_json:")
    print(json.dumps(parse_json(example.get("pii_spans_json"), []), indent=2)[:1200])
    print("\nmask_intervals_json:")
    print(json.dumps(parse_json(example.get("mask_intervals_json"), []), indent=2)[:800])


def span_bounds(span):
    """(start, end, label, text) from a PII span, whatever the key names are."""
    if not isinstance(span, dict):
        return None, None, str(span), ""
    start = span.get("start", span.get("start_char", span.get("begin")))
    end = span.get("end", span.get("end_char", span.get("stop")))
    label = span.get("label", span.get("entity", span.get("type", "PII")))
    text = span.get("text", span.get("value", ""))
    return start, end, label, text


def fmt_mask(m):
    """A silenced interval, printed as start-end seconds whatever its shape."""
    if isinstance(m, dict):
        start = m.get("start", m.get("start_sec", m.get("begin")))
        end = m.get("end", m.get("end_sec", m.get("stop")))
    elif isinstance(m, (list, tuple)) and len(m) >= 2:
        start, end = m[0], m[1]
    else:
        return str(m)
    def num(v):
        try:
            return f"{float(v):.2f}"
        except (TypeError, ValueError):
            return "?"
    return f"{num(start)}-{num(end)}s"


def _offset(v, limit):
    try:
        i = int(v)
    except (TypeError, ValueError):
        return None
    return i if 0 <= i <= limit else None


def highlight(text, spans):
    """Mark PII inside the transcript so you can see what should be inaudible.

    Only highlights when the offsets provably point at the span's own text. If
    pii_spans_json carries time offsets rather than character offsets, matching
    fails and the transcript is returned plain -- a convincing but wrong
    highlight is worse than none. Cell 6 lists every span regardless, so a
    failure here never hides PII from the reviewer.
    """
    text = str(text or "")
    marks = []
    for s in spans:
        start, end, label, claimed = span_bounds(s)
        start, end = _offset(start, len(text)), _offset(end, len(text))
        if start is None or end is None or start >= end:
            continue
        if claimed and text[start:end] != claimed:
            return text          # offsets disagree with the data; trust neither
        marks.append((start, end, label))
    out, cursor = [], 0
    for start, end, label in sorted(marks):
        if start < cursor:
            continue             # overlapping span, already covered by the previous mark
        out.append(text[cursor:start])
        out.append(
            f"<mark style='background:#ffd6d6;border:1px solid #d66'>{text[start:end]}"
            f"<sub style='color:#900'> {label}</sub></mark>"
        )
        cursor = end
    out.append(text[cursor:])
    return "".join(out)


def span_list(spans):
    """Every span as label + value, independent of whether offsets line up."""
    items = []
    for s in spans:
        start, end, label, claimed = span_bounds(s)
        items.append(f"<b>{label}</b>" + (f" {claimed}" if claimed else ""))
    return "<br>".join(items) or "&mdash;"

In [ ]:
# ---- 6. Review one call ----------------------------------------------------
def review(which):
    """Play ONE redacted call and show what was supposed to be removed from it.

    `which` is a call_id or its index in the list printed by cell 4.
    """
    s = BY_ID.get(which) if isinstance(which, str) else summary[which]
    if s is None:
        raise KeyError(f"{which!r} is not in this sample; re-run cell 4 for the id list")
    i = summary.index(s)
    segs = by_call[s["call_id"]]

    display(HTML(
        f"<h3 style='margin-bottom:2px'>[{i}] <code>{s['call_id']}</code></h3>"
        f"<div style='color:#555'><b>{s['call_sec']/60:.1f} min call</b> &middot; "
        f"{s['segments']} segments &middot; "
        f"<b>{s['pii_spans']} PII spans</b> &middot; {s['speech_sec']:.0f}s speech &middot; "
        f"status={s['status']} &middot; {s['size_kb']} KB</div>"
    ))

    if not s["exists"]:
        display(HTML("<b style='color:red'>redacted audio missing -- cannot verify or export</b>"))
        return

    display(HTML("<b>Redacted audio</b> (this is what ships to the vendor):"))
    data, sr = load_audio(s["path"])
    display(Audio(data, rate=sr))

    if PLAY_ORIGINAL:
        src = next((r.get("source_path_abs") for r in segs if r.get("source_path_abs")), None)
        if src and Path(src).exists():
            display(HTML("<b style='color:#a00'>ORIGINAL (contains PII)</b>:"))
            odata, osr = load_audio(src)
            display(Audio(odata, rate=osr))

    rows_html = []
    for r in segs:
        spans = parse_json(r.get("pii_spans_json"), [])
        masks = parse_json(r.get("mask_intervals_json"), [])
        if not spans and not masks:
            continue          # only show segments that were actually touched
        mask_txt = ", ".join(fmt_mask(m) for m in masks) or "&mdash;"
        rows_html.append(
            "<tr style='vertical-align:top'>"
            f"<td style='padding:3px 8px;white-space:nowrap'>ch{r.get('channel')} "
            f"{hms(r.get('start_sec'))}&ndash;{hms(r.get('end_sec'))}</td>"
            f"<td style='padding:3px 8px'>{highlight(r.get('final_transcript'), spans)}</td>"
            f"<td style='padding:3px 8px;font-size:12px'>{span_list(spans)}</td>"
            f"<td style='padding:3px 8px;white-space:nowrap;color:#06c'>{mask_txt}</td>"
            "</tr>"
        )

    display(HTML(
        "<table style='border-collapse:collapse;font-size:13px'>"
        "<tr><th style='text-align:left;padding:3px 8px'>time</th>"
        "<th style='text-align:left;padding:3px 8px'>transcript (PII highlighted)</th>"
        "<th style='text-align:left;padding:3px 8px'>PII found</th>"
        "<th style='text-align:left;padding:3px 8px'>mask intervals</th></tr>"
        + "".join(rows_html) + "</table>"
        "<div style='color:#555;margin-top:4px'>Listen at each mask interval: the "
        "highlighted words must be <b>inaudible</b>. Everything else must still be "
        "intelligible &mdash; over-masking is also a failure.</div><hr>"
    ))


# ---------------------------------------------------------------------------
# ONE call at a time. Every Audio() widget base64-embeds the whole decoded WAV
# into the notebook, so rendering 30 at once is tens of MB of output in a single
# document -- that is what makes the notebook crawl and eventually die.
#
# Paste an id from cell 4 (or an index), run this cell, listen, then change the
# id and run it again. Nothing accumulates: the previous player is replaced.
# ---------------------------------------------------------------------------
CALL = "0"        # <-- paste a call_id here, or leave an index like "0"/0

clear_output(wait=True)
review(int(CALL) if str(CALL).isdigit() else CALL)

In [ ]:
# ---- 7. Record your verdicts ----------------------------------------------
# Fill this in as you listen. Anything not marked "pass" is NOT staged.
#   "pass"       PII inaudible, rest of the call intelligible
#   "leak"       you could still hear something that should have been masked
#   "over"       masking removed non-PII speech
#   "bad_audio"  file corrupt / silent / wrong length
VERDICTS = {
    # "3201e9b6-c6f3-5a93-9e56-45d4ef952773": "pass",
}

REVIEWER = "sabarinath.g"

unreviewed = [s["call_id"] for s in summary if s["call_id"] not in VERDICTS]
tally = Counter(VERDICTS.values())
print(f"reviewed   : {len(VERDICTS)} / {len(summary)}")
for verdict, count in tally.most_common():
    print(f"  {verdict:10s} {count}")
if unreviewed:
    print(f"\nstill to review ({len(unreviewed)}):")
    for call_id in unreviewed:
        print(f'    "{call_id}": "",')

In [ ]:
# ---- 8. Stage approved calls ----------------------------------------------
import datetime as _dt
import shutil

approved = [s for s in summary if VERDICTS.get(s["call_id"]) == "pass" and s["exists"]]
leaks = [c for c, v in VERDICTS.items() if v == "leak"]

# Approved but with no file on disk: dropping these quietly would make the
# export smaller than the review says it is.
ghosts = [s["call_id"] for s in summary
          if VERDICTS.get(s["call_id"]) == "pass" and not s["exists"]]
if ghosts:
    print(f"!! {len(ghosts)} approved call(s) have no redacted file and will NOT be exported:")
    for call_id in ghosts:
        print("   ", call_id)

if leaks:
    raise SystemExit(
        f"{len(leaks)} call(s) marked as leaking PII: {leaks}\n"
        "Nothing staged. Fix the masking and re-run before any export."
    )
if not approved:
    raise SystemExit("no calls marked 'pass' -- nothing to stage")

# You listen to N_CALLS but ship at most EXPORT_N. Keep the highest-PII passes:
# among calls that all cleared review, those exercised the masking hardest and
# are the most useful evidence that it works.
surplus = []
if len(approved) > EXPORT_N:
    approved.sort(key=lambda s: -s["pii_spans"])
    approved, surplus = approved[:EXPORT_N], approved[EXPORT_N:]
    print(f"{len(surplus)} approved call(s) held back to keep the export at {EXPORT_N}:")
    for s in surplus:
        print(f"   {s['call_id']}  ({s['pii_spans']} PII)")
    print()

# Guard the rmtree below: only ever prune the run's own deid_results dir.
assert STAGE_DIR.name == "deid_results" and STAGE_DIR.is_relative_to(RUN_ROOT), STAGE_DIR
STAGE_DIR.mkdir(parents=True, exist_ok=True)

# The sync uploads every child of STAGE_DIR, so it must hold call dirs only.
strays = [p.name for p in STAGE_DIR.iterdir() if p.is_file()]
if strays:
    raise SystemExit(f"loose files in {STAGE_DIR} would be uploaded: {strays}")

# Drop call dirs left behind by an earlier review. `aws s3 sync` uploads what is
# in this directory, not what this session approved, so a leftover dir ships a
# call nobody verified today -- and if that call was previously rejected, it
# ships a known PII leak. The shell flow this replaces began with `rm -rf`.
approved_ids = {s["call_id"] for s in approved}
for path in sorted(STAGE_DIR.iterdir()):
    if path.is_dir() and path.name not in approved_ids:
        shutil.rmtree(path)
        print(f"removed stale staged call: {path.name}")

staged = []
for s in approved:
    dest_dir = STAGE_DIR / s["call_id"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / "audio.opus"
    shutil.copy2(s["path"], dest)
    size = dest.stat().st_size
    assert size == Path(s["path"]).stat().st_size, f"short copy for {s['call_id']}"
    staged.append({
        "call_id": s["call_id"],
        "source": s["path"],
        "bytes": size,
        "pii_spans": s["pii_spans"],
        "segments": s["segments"],
        "call_sec": round(s["call_sec"], 2),
        "speech_sec": round(s["speech_sec"], 2),
    })

# Audit record goes NEXT TO the stage dir, never inside it -- anything inside
# gets uploaded to the vendor bucket.
stamp = _dt.datetime.now(_dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
audit = STAGE_DIR.parent / f"verification_{stamp}.json"
audit.write_text(json.dumps({
    "run": RUN,
    "reviewer": REVIEWER,
    "verified_at_utc": stamp,
    "seed": SEED,
    "shards_sampled": [p.parts[-3] for p in MANIFESTS],
    "reviewed": len(VERDICTS),
    "verdicts": dict(Counter(VERDICTS.values())),
    "rejected": {c: v for c, v in VERDICTS.items() if v != "pass"},
    "approved_but_missing": ghosts,
    # Passed review but over the EXPORT_N cap -- not a rejection, and the first
    # place to draw from if the vendor asks for more.
    "held_back": [s["call_id"] for s in surplus],
    "staged": staged,
}, indent=2))

total_mb = sum(s["bytes"] for s in staged) / 1e6
print(f"staged {len(staged)} call(s), {total_mb:.1f} MB -> {STAGE_DIR}")
print(f"audit  {audit}")
for s in staged:
    print(f"  {s['call_id']}  {s['bytes']:>9,} B  {s['pii_spans']} PII")

In [ ]:
# ---- 9. Export commands ----------------------------------------------------
# Printed, not executed. Writing to the vendor bucket should be a deliberate
# shell action, and the assumed-role credentials must not land in notebook state.
# EXTERNAL_ID stays out of the repo -- export it in your shell first.
ROLE_ARN = "arn:aws:iam::170829523831:role/VendorDeidWriteRole"
BUCKET_PREFIX = "s3://amc-commercial-export/export/v1/deid_return"
KMS_KEY = "arn:aws:kms:us-east-1:170829523831:key/94db12d8-46e4-4ec4-b0a3-7e5c39b3ab74"

print(f"""# {len(staged)} verified call(s) from {RUN}, reviewed by {REVIEWER}
# expect exactly {len(staged)} objects and {sum(s['bytes'] for s in staged):,} bytes at the end

export EXTERNAL_ID="<from 1Password, do not paste into the notebook>"
export RUN_ID="run_$(date -u +%Y%m%dT%H%M%SZ)"
export LOCAL_DIR="{STAGE_DIR}"
export DEST="{BUCKET_PREFIX}/${{RUN_ID}}/"
export KMS_KEY="{KMS_KEY}"

ASSUME=$(aws sts assume-role \\
  --role-arn {ROLE_ARN} \\
  --role-session-name "oai-write-${{RUN_ID}}" \\
  --external-id "$EXTERNAL_ID" \\
  --duration-seconds 3600 \\
  --output json)
export AWS_ACCESS_KEY_ID=$(echo "$ASSUME" | jq -r '.Credentials.AccessKeyId')
export AWS_SECRET_ACCESS_KEY=$(echo "$ASSUME" | jq -r '.Credentials.SecretAccessKey')
export AWS_SESSION_TOKEN=$(echo "$ASSUME" | jq -r '.Credentials.SessionToken')
export AWS_DEFAULT_REGION=us-east-1

# Credential probe. It goes NEXT TO the run prefix, not inside it: a scratch file
# under $DEST would be delivered as part of the batch and would throw off the
# object count the final `ls` is there to confirm.
echo "write-test $(date -u)" > /tmp/amc-write-test.txt
aws s3 cp /tmp/amc-write-test.txt "{BUCKET_PREFIX}/write-test/amc-write-test.txt" \\
  --sse aws:kms --sse-kms-key-id "$KMS_KEY"

aws s3 sync "$LOCAL_DIR/" "$DEST" \\
  --sse aws:kms --sse-kms-key-id "$KMS_KEY" \\
  --only-show-errors

aws s3 ls "$DEST" --recursive --summarize""")